# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [44]:
import pandas as pd
import numpy as np

model_a = pd.read_csv('model-A.tsv', sep='\t')
model_b = pd.read_csv('model-B.tsv', sep='\t')
cost_r  = pd.read_csv('cost-R.tsv', sep='\t')
cost_q  = pd.read_csv('cost-Q.tsv', sep='\t')

In [2]:
model_a

,index,x,v,a
0,x,1,1,0
1,v,0,1,1
2,a,0,0,1


In [3]:
model_b

,index,u
0,x,0
1,v,0
2,a,1


In [4]:
cost_r

,index,u
0,u,5


In [45]:
cost_q

,index,x,v,a
0,x,5,0,0
1,v,0,1,0
2,a,0,0,2


In [46]:
import control

A = model_a[['x', 'v', 'a']].to_numpy()
B = model_b[['u']].to_numpy()
R = cost_r[['u']].to_numpy()
Q = cost_q[['x', 'v', 'a']].to_numpy()

K, S, E = control.dlqr(A, B, Q, R)
K

array([[0.33445985, 1.30445413, 1.85813088]])

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [47]:
K_df = pd.DataFrame(K, columns=['x', 'v', 'a'])
K_df.to_csv('control-K-intended.tsv', sep='\t', index=False)

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [48]:
log_states = pd.read_csv('qc-log.tsv', sep='\t')
log_states

,index,x,v,a,u
0,0,-5.000000e+00,0.000000e+00,0.000000e+00,1.702188e+00
1,1,-5.000000e+00,-1.702188e-02,1.531969e+00,-1.263200e+00
2,2,-5.018724e+00,1.452683e+00,5.482855e-01,-1.249321e+00
3,3,-3.420773e+00,1.840779e+00,-5.212749e-01,-2.121274e-01
4,4,-1.395916e+00,1.163611e+00,-7.643170e-01,4.528954e-01
...,...,...,...,...,...
95,95,-1.855920e-18,1.097460e-18,-1.592936e-19,-4.843645e-19
96,96,-6.487133e-19,8.412290e-19,-6.111510e-19,3.189634e-19
97,97,2.766386e-19,1.733230e-19,-3.851990e-19,4.316642e-19
98,98,4.672940e-19,-2.142650e-19,-3.522116e-20,1.881712e-19


In [49]:
log_states_np = log_states[['x', 'v', 'a']].to_numpy()

u_check = -log_states_np @ K.T

log_states['u_check'] = u_check
log_states

,index,x,v,a,u,u_check
0,0,-5.000000e+00,0.000000e+00,0.000000e+00,1.702188e+00,1.672299e+00
1,1,-5.000000e+00,-1.702188e-02,1.531969e+00,-1.263200e+00,-1.152095e+00
2,2,-5.018724e+00,1.452683e+00,5.482855e-01,-1.249321e+00,-1.235183e+00
3,3,-3.420773e+00,1.840779e+00,-5.212749e-01,-2.121274e-01,-2.885035e-01
4,4,-1.395916e+00,1.163611e+00,-7.643170e-01,4.528954e-01,3.692016e-01
...,...,...,...,...,...,...
95,95,-1.855920e-18,1.097460e-18,-1.592936e-19,-4.843645e-19,-5.148677e-19
96,96,-6.487133e-19,8.412290e-19,-6.111510e-19,3.189634e-19,2.552224e-19
97,97,2.766386e-19,1.733230e-19,-3.851990e-19,4.316642e-19,3.971337e-19
98,98,4.672940e-19,-2.142650e-19,-3.522116e-20,1.881712e-19,1.886533e-19


In [50]:
qc_check = log_states[['index', 'u_check']]
qc_check

,index,u_check
0,0,1.672299e+00
1,1,-1.152095e+00
2,2,-1.235183e+00
3,3,-2.885035e-01
4,4,3.692016e-01
...,...,...
95,95,-5.148677e-19
96,96,2.552224e-19
97,97,3.971337e-19
98,98,1.886533e-19


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [51]:
qc_check.to_csv('qc-check.tsv', sep='\t', index=False)

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [52]:
from sklearn.linear_model import LinearRegression

X = log_states[['x', 'v', 'a']].to_numpy()
y = log_states[['u']].to_numpy()

model = LinearRegression(fit_intercept=False)
model.fit(X, y)

# u = -Kx
coef = -model.coef_
coef

array([[0.34043755, 1.30012023, 1.95011696]])

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [53]:
coef_df = pd.DataFrame(coef, columns=['x', 'v', 'a'])
coef_df.to_csv('control-K-actual.tsv', sep='\t', index=False)

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [54]:
qc_train = pd.read_csv('qc-train.tsv', sep='\t')
qc_train

,index,x,v,a,u
0,0,-5.000000,0.000000,0.000000,1.729856
1,1,-5.000000,-0.017299,1.556871,-1.311911
2,2,-5.019028,1.476577,0.531837,-1.198850
3,3,-3.394793,1.846154,-0.493944,-0.297565
4,4,-1.364024,1.195267,-0.811147,0.472619
...,...,...,...,...,...
95,95,0.089539,-0.162778,0.122030,-0.008091
96,96,-0.089517,-0.030491,0.126952,-0.174120
97,97,-0.123056,0.094904,-0.017061,0.038104
98,98,-0.018662,0.068824,0.015527,-0.062828


In [55]:
X_t = qc_train[['x', 'v', 'a']].iloc[:-1].to_numpy() # all except last
U_t = qc_train[['u']].iloc[:-1].to_numpy()

# t+1 states
X_next = qc_train[['x', 'v', 'a']].iloc[1:].to_numpy()

Z = np.hstack([X_t, U_t])
model = LinearRegression(fit_intercept=False)
model.fit(Z, X_next)

coef = model.coef_

model_a_estimated = coef[:, :3] # (3, 3)
model_b_estimated = coef[:, 3:] # (3, 1)

In [56]:
# model_a_estimated
A_df = pd.DataFrame(model_a_estimated, columns=['x', 'v', 'a'])
A_df['index'] = ['x', 'v', 'a']
A_df = A_df[['index', 'x', 'v', 'a']]
A_df

,index,x,v,a
0,x,1.000000e+00,1.100000e+00,2.195949e-15
1,v,4.659919e-17,9.000000e-01,9.500000e-01
2,a,-6.756882e-16,-1.346180e-15,1.100000e+00


In [57]:
model_b_estimated
B_df = pd.DataFrame(model_b_estimated, columns=['u'])
B_df['index'] = ['x', 'v', 'a']
B_df = B_df[['index', 'u']]
B_df

,index,u
0,x,-9.514996e-17
1,v,-1.000000e-02
2,a,9.000000e-01


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [58]:
A_df.to_csv('model-A-new.tsv', sep='\t', index=False)
B_df.to_csv('model-B-new.tsv', sep='\t', index=False)

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.